In [0]:
%run ./00_config_setup

#### 1. S3 bucket path

#### All Utilities

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

def add_silver_metadata(df: DataFrame, domain: str) -> DataFrame:
    return (
        df
        .withColumn("silver_timestamp", current_timestamp())
        .withColumn("silver_batch_id",  lit(BATCH_ID))
        .withColumn("silver_domain",    lit(domain))
    )

def read_bronze(s3_path: str, domain: str) -> DataFrame:
    df = spark.read.format("delta").load(s3_path)
    return df


def merge_into_silver(df, table_name, merge_condition, domain):
    
    try:
        target = DeltaTable.forName(spark, table_name)
        (
            target.alias("t")
            .merge(df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[MERGE] {table_name}")

    except:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        print(f"[OVERWRITE] {table_name} — first load")

    cnt = spark.table(table_name).count()
    print(f"[COUNT] {table_name} → {cnt:,} rows")
    return cnt

def optimize_zorder(table_name: str, zorder_cols: list):
    zorder_str = ", ".join(zorder_cols)
    spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zorder_str})")
    

def show_detail(table_name: str):
    spark.sql(f"DESCRIBE DETAIL {table_name}").select(
        "name", "location", "numFiles", "sizeInBytes", "partitionColumns"
    ).show(truncate=False)


def show_history(table_name: str):
    spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 3").select(
        "version", "timestamp", "operation", "operationMetrics"
    ).show(truncate=False)


print("✅ All utilities ready")

✅ All utilities ready


#### 2. unity catalog path

#### 3. run identity

/home/spark-33779085-13c7-4af0-bd88-fd/.ipykernel/1857/command-8628948029972694-2015145426:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS         = datetime.utcnow()


#### DOMAIN 1 — ELECTRONICS SILVER

In [0]:
spark.sql("SHOW TABLES IN retailflow360.silver").show()

+--------+-----------------+-----------+
|database|        tableName|isTemporary|
+--------+-----------------+-----------+
|  silver|       books_data|      false|
|  silver|     books_rating|      false|
|  silver|      electronics|      false|
|  silver|     liquor_sales|      false|
|  silver|silver_checkpoint|      false|
+--------+-----------------+-----------+



In [0]:
df_elec = read_bronze(S3_BRONZE_ELECTRONICS, "electronics")

# Incremental Load
checkpoint_tbl = "silver_checkpoint"
if spark.catalog.tableExists(checkpoint_tbl):
    processed_batches = spark.table(checkpoint_tbl) \
        .filter(col("domain") == "electronics") \
        .select("batch_id")

    df_elec = df_elec.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )

# Drop embedded header rows
df_elec = df_elec.filter(
    col("order_id").isNotNull() &
    (col("order_id") != "Order ID") &
    (col("order_id") != "")
)

# Cast numerics
df_elec = (
    df_elec
    .withColumn("quantity_ordered_raw", col("quantity_ordered"))
    .withColumn("price_each_raw",       col("price_each"))
    .withColumn("quantity_ordered",     expr("try_cast(quantity_ordered as int)"))
    .withColumn("price_each",           expr("try_cast(price_each as double)"))
)

# Parse timestamp
df_elec = df_elec.withColumn(
    "order_timestamp",
    to_timestamp(col("order_date"), "MM/dd/yy H:mm")
)

# Temporal derivations
df_elec = (
    df_elec
    .withColumn("order_date_dt",  to_date(col("order_timestamp")))
    .withColumn("order_year",     year(col("order_timestamp")).cast(IntegerType()))
    .withColumn("order_month",    month(col("order_timestamp")).cast(IntegerType()))
    .withColumn("order_quarter",  quarter(col("order_timestamp")).cast(IntegerType()))
    .withColumn("order_hour",     hour(col("order_timestamp")).cast(IntegerType()))
    .withColumn("is_weekend",
        (dayofweek(col("order_timestamp")).isin(1, 7)).cast(BooleanType()))
    .withColumn("is_cross_year",
        (year(col("order_timestamp")) != 2019).cast(BooleanType()))
)



# Parse address
df_elec = (
    df_elec
    .withColumn("_parts",   split(trim(col("purchase_address")), ", "))
    .withColumn("street",   trim(col("_parts").getItem(0)))
    .withColumn("city",     trim(col("_parts").getItem(1)))
    .withColumn("_sz",      trim(col("_parts").getItem(2)))
    .withColumn("state",    regexp_extract(col("_sz"), r"^([A-Z]{2})", 1))
    .withColumn("zip_code", regexp_extract(col("_sz"), r"(\d{5})$", 1))
    .drop("_parts", "_sz")
)

# Line revenue
df_elec = df_elec.withColumn(
    "line_revenue",
    when(
        col("quantity_ordered").isNotNull() & col("price_each").isNotNull(),
        spark_round(col("quantity_ordered").cast(DoubleType()) * col("price_each"), 2)
    ).otherwise(lit(None).cast(DoubleType()))
)

# Duplicate order_id flag
w = Window.partitionBy("order_id")
df_elec = (
    df_elec
    .withColumn("order_id_count",
        F.count("order_id").over(w).cast(IntegerType()))
    .withColumn("is_duplicate_order_id",
        (col("order_id_count") > 1).cast(BooleanType()))
)


from pyspark.sql.functions import col, when, lit, concat_ws

# Quality flags
df_elec = (
    df_elec
    .withColumn("has_valid_quantity",
        (col("quantity_ordered").isNotNull() & (col("quantity_ordered") > 0)).cast(BooleanType()))
    .withColumn("has_valid_price",
        (col("price_each").isNotNull() & (col("price_each") > 0)).cast(BooleanType()))
    .withColumn("has_valid_timestamp",
        col("order_timestamp").isNotNull().cast(BooleanType()))
    .withColumn("has_valid_address",
        col("state").isNotNull() & (col("state") != "").cast(BooleanType()))
    .withColumn(
        "row_quality_flag",
        concat_ws(",",
            when(~col("has_valid_quantity"),   lit("NULL_OR_INVALID_QUANTITY")),
            when(~col("has_valid_price"),     lit("NULL_OR_INVALID_PRICE")),
            when(~col("has_valid_timestamp"), lit("NULL_TIMESTAMP")),
            when(~col("has_valid_address"),   lit("BAD_ADDRESS")),
            when(col("is_cross_year"),        lit("CROSS_YEAR"))
        )
    )
    .withColumn(
        "row_quality_flag",
        when(col("row_quality_flag") == "", lit("CLEAN"))
        .otherwise(col("row_quality_flag"))
    )
)



df_elec = df_elec.drop("order_date")
df_elec = add_silver_metadata(df_elec, "electronics")

# MERGE into Silver 

df_elec = df_elec.dropDuplicates(["order_id"])
elec_rows = merge_into_silver(
    df            = df_elec,
    table_name    = TBL_SLV_ELECTRONICS,
    merge_condition = "t.order_id = s.order_id AND t.product = s.product",
    domain        = "electronics"
)


# processed batch_ids
df_elec.select("batch_id").distinct() \
    .withColumn("domain", lit("electronics")) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .write.mode("append").saveAsTable(checkpoint_tbl)


optimize_zorder(TBL_SLV_ELECTRONICS, ["order_id"])
show_history(TBL_SLV_ELECTRONICS)
show_detail(TBL_SLV_ELECTRONICS)

print("\nSample:")
display(spark.table(TBL_SLV_ELECTRONICS).limit(5))

print("\nQuality distribution:")
spark.table(TBL_SLV_ELECTRONICS) \
    .groupBy("row_quality_flag").count() \
    .orderBy(col("count").desc()).show()

+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                

batch_id,order_id,product,quantity_ordered,price_each,purchase_address,ingestion_timestamp,ingestion_date,source_file_name,domain,source_month,quantity_ordered_raw,price_each_raw,order_timestamp,order_date_dt,order_year,order_month,order_quarter,order_hour,is_weekend,is_cross_year,street,city,state,zip_code,line_revenue,order_id_count,is_duplicate_order_id,has_valid_quantity,has_valid_price,has_valid_timestamp,has_valid_address,row_quality_flag,silver_timestamp,silver_batch_id,silver_domain
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,141240,27in 4K Gaming Monitor,1,389.99,"979 Park St, Los Angeles, CA 90001",2026-04-22T08:38:20.003Z,2026-04-22,s3://retailflow360-capstone3/raw/electronics/,electronics,January_2019,1,389.99,2019-01-26T12:16:00.000Z,2019-01-26,2019,1,1,12,true,false,979 Park St,Los Angeles,CA,90001,389.99,2,true,true,true,true,true,CLEAN,2026-04-22T09:47:15.435Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,electronics
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,141241,USB-C Charging Cable,1,11.95,"181 6th St, San Francisco, CA 94016",2026-04-22T08:38:20.003Z,2026-04-22,s3://retailflow360-capstone3/raw/electronics/,electronics,January_2019,1,11.95,2019-01-05T12:04:00.000Z,2019-01-05,2019,1,1,12,true,false,181 6th St,San Francisco,CA,94016,11.95,2,true,true,true,true,true,CLEAN,2026-04-22T09:47:15.435Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,electronics
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,141251,Apple Airpods Headphones,1,150.0,"414 Walnut St, Boston, MA 02215",2026-04-22T08:38:20.003Z,2026-04-22,s3://retailflow360-capstone3/raw/electronics/,electronics,January_2019,1,150,2019-01-24T08:13:00.000Z,2019-01-24,2019,1,1,8,false,false,414 Walnut St,Boston,MA,02215,150.0,2,true,true,true,true,true,CLEAN,2026-04-22T09:47:15.435Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,electronics
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,141253,AA Batteries (4-pack),1,3.84,"385 11th St, Atlanta, GA 30301",2026-04-22T08:38:20.003Z,2026-04-22,s3://retailflow360-capstone3/raw/electronics/,electronics,January_2019,1,3.84,2019-01-17T00:09:00.000Z,2019-01-17,2019,1,1,0,false,false,385 11th St,Atlanta,GA,30301,3.84,2,true,true,true,true,true,CLEAN,2026-04-22T09:47:15.435Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,electronics
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,141256,Google Phone,1,600.0,"675 Washington St, Portland, OR 97035",2026-04-22T08:38:20.003Z,2026-04-22,s3://retailflow360-capstone3/raw/electronics/,electronics,January_2019,1,600,2019-01-29T10:40:00.000Z,2019-01-29,2019,1,1,10,false,false,675 Washington St,Portland,OR,97035,600.0,2,true,true,true,true,true,CLEAN,2026-04-22T09:47:15.435Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,electronics



Quality distribution:
+----------------+------+
|row_quality_flag| count|
+----------------+------+
|           CLEAN|178406|
|      CROSS_YEAR|    31|
+----------------+------+



#### DOMAIN 2 — LIQUOR SALES SILVER

In [0]:
df_liq = read_bronze(S3_BRONZE_LIQUOR, "liquor")

checkpoint_tbl = "silver_checkpoint"
if spark.catalog.tableExists(checkpoint_tbl):
    processed_batches = spark.table(checkpoint_tbl) \
        .filter(col("domain") == "liquor") \
        .select("batch_id")

    df_liq = df_liq.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )

# Parse date
df_liq = df_liq.withColumn(
    "transaction_date",
    coalesce(
        expr("try_to_date(date, 'MM/dd/yyyy')"),
        expr("try_to_date(date, 'yyyy-MM-dd')"),
        expr("try_to_date(date, 'M/d/yyyy')")
    )
)

# Temporal
df_liq = (
    df_liq
    .withColumn("transaction_year",    year(col("transaction_date")).cast(IntegerType()))
    .withColumn("transaction_month",   month(col("transaction_date")).cast(IntegerType()))
    .withColumn("transaction_quarter", quarter(col("transaction_date")).cast(IntegerType()))
)

# WKT POINT → lon, lat
df_liq = (
    df_liq
    .withColumn("lon",
        when(col("store_location").isNotNull() & (col("store_location") != ""),
            regexp_extract(col("store_location"),
                r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)", 1).cast(DoubleType())
        ).otherwise(lit(None).cast(DoubleType()))
    )
    .withColumn("lat",
        when(col("store_location").isNotNull() & (col("store_location") != ""),
            regexp_extract(col("store_location"),
                r"POINT\s*\(\s*([-\d.]+)\s+([-\d.]+)\s*\)", 2).cast(DoubleType())
        ).otherwise(lit(None).cast(DoubleType()))
    )
    .withColumn("has_geo",
        (col("store_location").isNotNull() &
         (col("store_location") != "")).cast(BooleanType()))
)

# Geo quality flag
df_liq = df_liq.withColumn(
    "geo_quality",
    when(~col("has_geo"), lit("NO_GEO"))
    .when(
        col("lat").isNotNull() & col("lon").isNotNull() &
        ((col("lat") < 40.37) | (col("lat") > 43.50) |
         (col("lon") < -96.64) | (col("lon") > -90.14)),
        lit("SUSPECT_COORDS")
    )
    .otherwise(lit("VALID"))
)

# Title case city, county
df_liq = (
    df_liq
    .withColumn("city",   initcap(trim(col("city"))))
    .withColumn("county", initcap(trim(col("county"))))
)

# Clean dollar columns
for dc in ["sale__dollars_", "state_bottle_cost", "state_bottle_retail"]:
    if dc in df_liq.columns:
        df_liq = df_liq.withColumn(
            dc, expr(f"try_cast(regexp_replace({dc}, '[\\$,]', '') as double)")
        )

# Cast numerics
for c, t in {
    "store_number": "int", "county_number": "int",
    "vendor_number": "int", "item_number": "int",
    "pack": "int", "bottle_volume__ml_": "int",
    "bottles_sold": "int", "volume_sold__liters_": "double",
    "volume_sold__gallons_": "double"
}.items():
    if c in df_liq.columns:
        df_liq = df_liq.withColumn(c, expr(f"try_cast({c} as {t})"))

# Pad 4-digit zips
if "zip_code" in df_liq.columns:
    df_liq = df_liq.withColumn(
        "zip_code",
        when(
            col("zip_code").isNotNull() &
            col("zip_code").rlike(r"^\d{4}$"),
            F.concat(lit("0"), col("zip_code"))
        ).otherwise(col("zip_code"))
    )

# Title case store name
if "store_name" in df_liq.columns:
    df_liq = df_liq.withColumn("store_name", initcap(trim(col("store_name"))))

# Sale matches calc
df_liq = df_liq.withColumn(
    "sale_matches_calc",
    when(
        col("bottles_sold").isNotNull() &
        col("state_bottle_retail").isNotNull() &
        col("sale__dollars_").isNotNull() &
        (col("state_bottle_retail") > 0),
        (
            F.abs(col("sale__dollars_") -
            spark_round(col("bottles_sold").cast(DoubleType()) *
                        col("state_bottle_retail"), 2))
            <= (col("sale__dollars_") * 0.05)
        ).cast(BooleanType())
    ).otherwise(lit(None).cast(BooleanType()))
)

# Row quality flag
df_liq = df_liq.withColumn(
    "row_quality_flag",
    when(
        col("invoice_item_number").isNull() |
        (col("invoice_item_number") == ""),
        lit("CRITICAL_NULL_PK"))
    .when(col("transaction_date").isNull(), lit("FLAGGED_NULL_DATE"))
    .when(
        col("bottles_sold").isNotNull() & (col("bottles_sold") > 10000),
        lit("FLAGGED_HIGH_VOLUME"))
    .when(
        col("bottles_sold").isNotNull() & (col("bottles_sold") == 0),
        lit("FLAGGED_ZERO_BOTTLES"))
    .when(
        col("state_bottle_cost").isNotNull() &
        col("state_bottle_retail").isNotNull() &
        (col("state_bottle_retail") < col("state_bottle_cost")),
        lit("FLAGGED_INVERTED_MARGIN"))
    .when(
        col("state_bottle_cost").isNotNull() &
        (col("state_bottle_cost") == 0),
        lit("FLAGGED_ZERO_PRICE"))
    .when(
        col("sale__dollars_").isNull() | (col("sale__dollars_") == 0),
        lit("FLAGGED_ZERO_SALE"))
    .otherwise(lit("CLEAN"))
)

df_liq = add_silver_metadata(df_liq, "liquor")

# MERGE
df_liq = df_liq.dropDuplicates(["invoice_item_number"])
liq_rows = merge_into_silver(
    df              = df_liq,
    table_name      = TBL_SLV_LIQUOR,
    merge_condition = "t.invoice_item_number = s.invoice_item_number",
    domain          = "liquor"
)


df_liq.select("batch_id").distinct() \
    .withColumn("domain", lit("liquor")) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .write.mode("append").saveAsTable(checkpoint_tbl)


optimize_zorder(TBL_SLV_LIQUOR, ["invoice_item_number", "transaction_date"])
show_history(TBL_SLV_LIQUOR)
show_detail(TBL_SLV_LIQUOR)

print("\nSample:")
display(spark.table(TBL_SLV_LIQUOR).limit(5))

print("\nQuality distribution:")
spark.table(TBL_SLV_LIQUOR) \
    .groupBy("row_quality_flag").count() \
    .orderBy(col("count").desc()).show()

+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                   

batch_id,invoice_item_number,date,store_number,store_name,address,city,zip_code,store_location,county_number,county,category,category_name,vendor_number,vendor_name,item_number,item_description,pack,bottle_volume__ml_,state_bottle_cost,state_bottle_retail,bottles_sold,sale__dollars_,volume_sold__liters_,volume_sold__gallons_,_raw_file_path,ingestion_timestamp,ingestion_date,source_file_name,domain,transaction_date,transaction_year,transaction_month,transaction_quarter,lon,lat,has_geo,geo_quality,sale_matches_calc,row_quality_flag,silver_timestamp,silver_batch_id,silver_domain
5658f921-d318-40ee-8c4b-1a5ba5c57c88,INV-30253800008,09/15/2020,5089,Mos Mini Mart,"614, E Erie St",Missouri Valley,51555,POINT (-95.88687 41.556439),43,Harrison,1052100,Imported Brandies,420,MOET HENNESSY USA,48105,Hennessy VS,12,375,10.99,16.49,2,32.98,0.75,0.19,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T09:31:42.864Z,2026-04-22,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor,2020-09-15,2020,9,3,-95.88687,41.556439,true,VALID,true,CLEAN,2026-04-22T09:50:25.768Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,liquor
5658f921-d318-40ee-8c4b-1a5ba5c57c88,INV-30253800009,09/15/2020,5089,Mos Mini Mart,"614, E Erie St",Missouri Valley,51555,POINT (-95.88687 41.556439),43,Harrison,1062500,Flavored Rum,260,DIAGEO AMERICAS,72927,Captain Morgan Sliced Apple Mini,12,50,4.8,7.2,12,86.4,0.6,0.15,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T09:31:42.864Z,2026-04-22,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor,2020-09-15,2020,9,3,-95.88687,41.556439,true,VALID,true,CLEAN,2026-04-22T09:50:25.768Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,liquor
5658f921-d318-40ee-8c4b-1a5ba5c57c88,INV-30254500005,09/15/2020,4879,Sam's Mini Mart / Morningside Ave Sioux City,4218 MORNINGSIDE AVE,Sioux City,51106,POINT (-96.352851 42.47063),97,Woodbury,1081600,Whiskey Liqueur,421,SAZERAC COMPANY INC,64864,Fireball Cinnamon Whisky,24,375,5.33,8.0,24,192.0,9.0,2.37,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T09:31:42.864Z,2026-04-22,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor,2020-09-15,2020,9,3,-96.352851,42.47063,true,VALID,true,CLEAN,2026-04-22T09:50:25.768Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,liquor
5658f921-d318-40ee-8c4b-1a5ba5c57c88,INV-30254500011,09/15/2020,4879,Sam's Mini Mart / Morningside Ave Sioux City,4218 MORNINGSIDE AVE,Sioux City,51106,POINT (-96.352851 42.47063),97,Woodbury,1081600,Whiskey Liqueur,421,SAZERAC COMPANY INC,65013,Fireball Cinnamon Whiskey 50ml Sleeve,12,50,5.0,7.5,48,360.0,2.4,0.63,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T09:31:42.864Z,2026-04-22,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor,2020-09-15,2020,9,3,-96.352851,42.47063,true,VALID,true,CLEAN,2026-04-22T09:50:25.768Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,liquor
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,INV-30254500022,09/15/2020,4879,Sam's Mini Mart / Morningside Ave Sioux City,4218 MORNINGSIDE AVE,Sioux City,51106,POINT (-96.352851 42.47063),97,Woodbury,1041100,American Dry Gins,370,PERNOD RICARD USA,32232,Seagrams Extra Dry Gin,48,100,0.97,1.46,48,70.08,4.8,1.26,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,2026-04-22T08:38:50.801Z,2026-04-22,s3://retailflow360-capstone3/raw/liquor/Liquor_Sales.csv,liquor,2020-09-15,2020,9,3,-96.352851,42.47063,true,VALID,true,CLEAN,2026-04-22T09:50:25.768Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,liquor



Quality distribution:
+--------------------+--------+
|    row_quality_flag|   count|
+--------------------+--------+
|               CLEAN|19661757|
|  FLAGGED_ZERO_PRICE|    3084|
|   FLAGGED_ZERO_SALE|    1868|
|FLAGGED_INVERTED_...|      41|
|FLAGGED_ZERO_BOTTLES|       9|
| FLAGGED_HIGH_VOLUME|       4|
+--------------------+--------+



#### DOMAIN 3 — BOOKS DATA SILVER

In [0]:
df_bd = read_bronze(S3_BRONZE_BOOKS_DATA, "books_data")



checkpoint_tbl = "silver_checkpoint"
if spark.catalog.tableExists(checkpoint_tbl):
    processed_batches = spark.table(checkpoint_tbl) \
        .filter(col("domain") == "books_data") \
        .select("batch_id") \
        .distinct()

    df_bd = df_bd.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )




# Primary author — aggressive bracket/quote stripping
df_bd = df_bd.withColumn(
    "_ac",
    when(col("authors").isNotNull() & (col("authors") != ""),
        regexp_replace(regexp_replace(col("authors"), r"^\[|\]$", ""), r"^\s*|\s*$", "")
    ).otherwise(lit(None).cast(StringType()))
)
df_bd = df_bd.withColumn(
    "primary_author",
    when(col("_ac").isNotNull(),
        trim(regexp_replace(
            regexp_extract(col("_ac"), r"^([^,]+)", 1),
            r"['\"\[\]\\u]|\\\\", ""
        ))
    ).otherwise(lit(None).cast(StringType()))
)
df_bd = df_bd.withColumn(
    "primary_author",
    when(
        col("primary_author").isNull() |
        (trim(col("primary_author")) == "") |
        (length(trim(col("primary_author"))) < 2),
        lit(None).cast(StringType())
    ).otherwise(trim(col("primary_author")))
).drop("_ac")

# Primary category
df_bd = df_bd.withColumn(
    "primary_category",
    when(col("categories").isNotNull() & (col("categories") != ""),
        trim(regexp_replace(
            regexp_extract(
                regexp_replace(col("categories"), r"^\[|\]$", ""),
                r"^([^,]+)", 1
            ),
            r"['\"\[\]\\]", ""
        ))
    ).otherwise(lit(None).cast(StringType()))
)
df_bd = df_bd.withColumn(
    "primary_category",
    when(
        col("primary_category").isNull() | (trim(col("primary_category")) == ""),
        lit("Uncategorized")
    ).otherwise(trim(col("primary_category")))
)

# Published date — 6 format coalesce
df_bd = df_bd.withColumn(
    "published_date",
    coalesce(
        expr("try_to_date(publisheddate, 'yyyy-MM-dd')"),
        expr("try_to_date(publisheddate, 'MM/dd/yyyy')"),
        expr("try_to_date(publisheddate, 'dd-MMM-yy')"),
        expr("try_to_date(concat(publisheddate, '-01'), 'yyyy-MM-dd')"),
        expr("try_to_date(concat(publisheddate, '-01-01'), 'yyyy-MM-dd')"),
        expr("""try_to_date(
            regexp_extract(publisheddate, '(\\d{4}-\\d{2}-\\d{2})', 1),
            'yyyy-MM-dd')""")
    )
)
df_bd = df_bd.withColumn(
    "published_year", year(col("published_date")).cast(IntegerType())
)

# Year plausibility
df_bd = df_bd.withColumn(
    "year_plausibility_flag",
    when(col("published_year").isNull(),   lit("UNKNOWN_YEAR"))
    .when(col("published_year") < 1800,    lit("SUSPECT_TOO_OLD"))
    .when(col("published_year") > 2025,    lit("SUSPECT_FUTURE"))
    .otherwise(lit("VALID"))
)




# Ratings count cast
df_bd = df_bd.withColumn(
    "ratings_count",
    col("ratingscount").cast("double").cast("int")
)


# Strip control chars + truncate description

df_bd = df_bd.withColumn(
    "description",
    when(col("description").isNotNull(),
        regexp_replace(col("description"),
            r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "")
    ).otherwise(lit(None).cast(StringType()))
)
df_bd = df_bd.withColumn(
    "description",
    when(
        col("description").isNotNull() & (length(col("description")) > 10000),
        col("description").substr(1, 10000)
    ).otherwise(col("description"))
)

# Trim
df_bd = (
    df_bd
    .withColumn("title",     trim(col("title")))
    .withColumn("publisher", trim(col("publisher")))
)


# Presence flags
df_bd = (
    df_bd
    .withColumn("has_description",
        (col("description").isNotNull() & (col("description") != "")).cast(BooleanType()))
    .withColumn("has_author",
        col("primary_author").isNotNull().cast(BooleanType()))
    .withColumn("has_publisher",
        col("publisher").isNotNull().cast(BooleanType()))
)

# Row quality flag
df_bd = df_bd.withColumn(
    "row_quality_flag",
    when(col("title").isNull() | (col("title") == ""), lit("CRITICAL_NULL_PK"))
    .when(col("year_plausibility_flag") == "SUSPECT_FUTURE", lit("FLAGGED_FUTURE_YEAR"))
    .when(col("year_plausibility_flag") == "SUSPECT_TOO_OLD", lit("FLAGGED_OLD_YEAR"))
    .when(~col("has_description"), lit("FLAGGED_NULL_DESC"))
    .otherwise(lit("CLEAN"))
)

df_bd = add_silver_metadata(df_bd, "books_data")

# MERGE

df_bd = df_bd.dropDuplicates(["title"])
bd_rows = merge_into_silver(
    df              = df_bd,
    table_name      = TBL_SLV_BOOKS_DATA,
    merge_condition = "t.title = s.title AND t.primary_author = s.primary_author",
    domain          = "books_data"
)

df_bd.select("batch_id").distinct() \
    .withColumn("domain", lit("books_data")) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .write.mode("append").saveAsTable(checkpoint_tbl)



optimize_zorder(TBL_SLV_BOOKS_DATA, ["title"])
show_history(TBL_SLV_BOOKS_DATA)
show_detail(TBL_SLV_BOOKS_DATA)

print("\nSample:")
display(spark.table(TBL_SLV_BOOKS_DATA).limit(5))

print("\nQuality distribution:")
spark.table(TBL_SLV_BOOKS_DATA) \
    .groupBy("row_quality_flag").count() \
    .orderBy(col("count").desc()).show()

+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                              

batch_id,title,description,authors,image,previewlink,publisher,publisheddate,infolink,categories,ratingscount,_raw_file_path,ingestion_timestamp,ingestion_date,source_file_name,domain,primary_author,primary_category,published_date,published_year,year_plausibility_flag,ratings_count,has_description,has_author,has_publisher,row_quality_flag,silver_timestamp,silver_batch_id,silver_domain
5658f921-d318-40ee-8c4b-1a5ba5c57c88,"""... And Poetry is Born ..."" Russian Classical Poetry","A selection of Russian poems in Russian with an English translation, with works from Pushkin, Lermontov, Tyutchev, Bunin, Blok, Akhmatova, Mayakovsky.",['Aleksandr Sergeevich Pushkin'],null,http://books.google.nl/books?id=IWRhwgEACAAJ&dq=%22...+And+Poetry+is+Born+...%22+Russian+Classical+Poetry&hl=&cd=1&source=gbs_api,null,1984,http://books.google.nl/books?id=IWRhwgEACAAJ&dq=%22...+And+Poetry+is+Born+...%22+Russian+Classical+Poetry&hl=&source=gbs_api,['Russian poetry'],null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T09:32:50.599Z,2026-04-22,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data,Aleksandr Sergeevich Pshkin,Russian poetry,1984-01-01,1984,VALID,null,true,true,false,CLEAN,2026-04-22T09:53:43.619Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_data
5658f921-d318-40ee-8c4b-1a5ba5c57c88,"""Henry Herbes, come on over"": A history of the German Emigrant, Henry Herbes, 1840-1901",null,['Eleanor Herbes Wollenzien'],null,"http://books.google.nl/books?id=RkgNHQAACAAJ&dq=%22Henry+Herbes,+come+on+over%22:+A+history+of+the+German+Emigrant,+Henry+Herbes,+1840-1901&hl=&cd=1&source=gbs_api",null,1988,"http://books.google.nl/books?id=RkgNHQAACAAJ&dq=%22Henry+Herbes,+come+on+over%22:+A+history+of+the+German+Emigrant,+Henry+Herbes,+1840-1901&hl=&source=gbs_api",null,null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T09:32:50.599Z,2026-04-22,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data,Eleanor Herbes Wollenzien,Uncategorized,1988-01-01,1988,VALID,null,false,true,false,FLAGGED_NULL_DESC,2026-04-22T09:53:43.619Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_data
5658f921-d318-40ee-8c4b-1a5ba5c57c88,*OP Clanbook: Ravnos (Revised Ed) (Vampire: The Masquerade Clanbooks),"This sourcebook for Vampire: the masquerade includes details of Gangrel unlife, plus new secrets, Discipline powers and clan lore.","['Brian Campbell', 'James Kiley', 'Ellen Kiley']",http://books.google.com/books/content?id=dzUIAAAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api,http://books.google.com/books?id=dzUIAAAACAAJ&dq=*OP+Clanbook:+Ravnos+(Revised+Ed)+(Vampire:+The+Masquerade+Clanbooks)&hl=&cd=1&source=gbs_api,White Wolf Pub,2000-07-01,http://books.google.com/books?id=dzUIAAAACAAJ&dq=*OP+Clanbook:+Ravnos+(Revised+Ed)+(Vampire:+The+Masquerade+Clanbooks)&hl=&source=gbs_api,['Fiction'],null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T09:32:50.599Z,2026-04-22,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data,Brian Campbell,Fiction,2000-07-01,2000,VALID,null,true,true,true,CLEAN,2026-04-22T09:53:43.619Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_data
5658f921-d318-40ee-8c4b-1a5ba5c57c88,"*OP Risen, The",null,['James Plunkett'],null,"http://books.google.nl/books?id=99gDPwAACAAJ&dq=*OP+Risen,+The&hl=&cd=1&source=gbs_api",null,1978,"http://books.google.nl/books?id=99gDPwAACAAJ&dq=*OP+Risen,+The&hl=&source=gbs_api",null,null,s3://retailflow360-capstone3/raw/books/books_data.csv,2026-04-22T09:32:50.599Z,2026-04-22,s3://retailflow360-capstone3/raw/books/books_data.csv,books_data,James Plnkett,Uncategorized,1978-01-01,1978,VALID,null,false,true,false,FLAGGED_NULL_DESC,2026-04-22T09:53:43.619Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_data
5658f921-d318-40ee-8c4b-1a5ba5c57c88,1 & 2 Timothy and Titus: Letters to two young men (Teach yourself the Bible series),"Contains quotations, reflection questions, and stories designed to help individuals figure out how they can make a difference and create li


Quality distribution:
+-------------------+------+
|   row_quality_flag| count|
+-------------------+------+
|              CLEAN|143950|
|  FLAGGED_NULL_DESC| 68255|
|   FLAGGED_OLD_YEAR|   197|
|FLAGGED_FUTURE_YEAR|     1|
|   CRITICAL_NULL_PK|     1|
+-------------------+------+



#### DOMAIN 4 — BOOKS RATING SILVER

In [0]:
df_br = read_bronze(S3_BRONZE_BOOKS_RATING, "books_rating")

checkpoint_tbl = "silver_checkpoint"
if spark.catalog.tableExists(checkpoint_tbl):
    processed_batches = spark.table(checkpoint_tbl) \
        .filter(col("domain") == "books_rating") \
        .select("batch_id") \
        .distinct()

    df_br = df_br.join(
        processed_batches,
        on="batch_id",
        how="left_anti"
    )

# Parse epoch → date
# Epoch time = number of seconds since a fixed starting point

df_br = df_br.filter(col("review_time").cast("long") > 0)

df_br = (
    df_br
    .withColumn("_epoch",       col("review_time").cast(LongType()))
    .withColumn("review_date",  to_date(from_unixtime(col("_epoch"))))
    .withColumn("review_year",  year(col("review_date")).cast(IntegerType()))
    .withColumn("review_month", month(col("review_date")).cast(IntegerType()))
    .drop("_epoch")
)

# True deduplication — dedup on 6-column composite key
TRUE_DUP_COLS = [
    "id", "user_id", "review_time",
    "review_score", "review_helpfulness", "review_summary"
]
w_dedup = (
    Window
    .partitionBy(TRUE_DUP_COLS)
    .orderBy(F.length(col("review_text")).desc(), col("review_time").desc())
)
df_br = (
    df_br
    .withColumn("_rank", F.row_number().over(w_dedup))
    .filter(col("_rank") == 1)
    .drop("_rank")
)

# Rename id → book_id + create surrogate review_id
df_br = df_br.withColumnRenamed("id", "book_id")
df_br = df_br.withColumn(
    "review_id",
    md5(concat_ws("|",
        col("book_id"), col("user_id"), col("review_time"),
        col("review_score"), col("review_helpfulness"),
        F.coalesce(col("review_summary"), lit(""))
    ))
)

# Normalize user_id
df_br = df_br.withColumn(
    "user_id",
    when(
        col("user_id").isNull() |
        (trim(col("user_id")) == "") |
        lower(col("user_id")).rlike(r"^anon"),
        lit("ANONYMOUS")
    ).otherwise(trim(col("user_id")))
)

# Parse helpfulness fraction
df_br = (
    df_br
    .withColumn("helpful_votes_raw",
        regexp_extract(col("review_helpfulness"), r"^(\d+)/", 1).cast(IntegerType()))
    .withColumn("total_votes",
        regexp_extract(col("review_helpfulness"), r"/(\d+)$", 1).cast(IntegerType()))
)

# Cap helpful at total
df_br = df_br.withColumn(
    "helpful_votes",
    when(
        col("helpful_votes_raw").isNotNull() &
        col("total_votes").isNotNull() &
        (col("helpful_votes_raw") > col("total_votes")),
        col("total_votes")
    ).otherwise(col("helpful_votes_raw"))
).drop("helpful_votes_raw")

# Helpfulness ratio
df_br = df_br.withColumn(
    "helpfulness_ratio",
    when(
        col("total_votes").isNotNull() & (col("total_votes") > 0),
        spark_round(
            col("helpful_votes").cast(DoubleType()) /
            col("total_votes").cast(DoubleType()), 4)
    ).otherwise(lit(None).cast(DoubleType()))
)

# Cast review_score
df_br = df_br.withColumn("review_score", col("review_score").cast(DoubleType()))
df_br = df_br.withColumn(
    "is_valid_score",
    (col("review_score").isNotNull() &
     (col("review_score") >= 1.0) &
     (col("review_score") <= 5.0)).cast(BooleanType())
)

# Cast price
df_br = df_br.withColumn("price", col("price").cast(DoubleType()))

# Strip control chars + truncate review text
df_br = df_br.withColumn(
    "review_text_truncated",
    when(col("review_text").isNotNull(),
        regexp_replace(col("review_text"),
            r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "").substr(1, 5000)
    ).otherwise(lit(None).cast(StringType()))
)

df_br = df_br.withColumn(
    "review_text_was_truncated",
    (col("review_text").isNotNull() &
     (F.length(col("review_text")) > 5000)).cast(BooleanType())
)


df_br = df_br.withColumn("profilename", trim(col("profilename")))


df_br = df_br.withColumn(
    "is_valid_review_date",
    (col("review_date").isNotNull() &
     (col("review_date") >= F.to_date(lit("1995-01-01"))) &
     (col("review_date") <= F.to_date(lit("2023-12-31")))).cast(BooleanType())
)

# Row quality flag
df_br = df_br.withColumn(
    "row_quality_flag",
    when(col("book_id").isNull() | (col("book_id") == ""), lit("CRITICAL_NULL_PK"))
    .when(col("title").isNull() | (col("title") == ""), lit("FLAGGED_NULL_TITLE"))
    .when(~col("is_valid_score"), lit("FLAGGED_INVALID_SCORE"))
    .when(~col("is_valid_review_date"), lit("FLAGGED_INVALID_DATE"))
    .when(col("user_id") == "ANONYMOUS", lit("FLAGGED_ANONYMOUS_USER"))
    .otherwise(lit("CLEAN"))
)



df_br = add_silver_metadata(df_br, "books_rating")

# MERGE on review_id
df_br = df_br.dropDuplicates(["review_id"]) 
br_rows = merge_into_silver(
    df              = df_br,
    table_name      = TBL_SLV_BOOKS_RATING,
    merge_condition = "t.review_id = s.review_id",
    domain          = "books_rating"
)

df_br.select("batch_id").distinct() \
    .withColumn("domain", lit("books_rating")) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .write.mode("append").saveAsTable(checkpoint_tbl)



optimize_zorder(TBL_SLV_BOOKS_RATING, ["book_id", "review_id"])
show_history(TBL_SLV_BOOKS_RATING)
show_detail(TBL_SLV_BOOKS_RATING)

print("\nSample:")
display(spark.table(TBL_SLV_BOOKS_RATING).limit(5))

print("\nQuality distribution:")
spark.table(TBL_SLV_BOOKS_RATING) \
    .groupBy("row_quality_flag").count() \
    .orderBy(col("count").desc()).show()

+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                       

batch_id,book_id,title,price,user_id,profilename,review_helpfulness,review_score,review_time,review_summary,review_text,_raw_file_path,ingestion_timestamp,ingestion_date,source_file_name,domain,review_date,review_year,review_month,review_id,total_votes,helpful_votes,helpfulness_ratio,is_valid_score,review_text_truncated,review_text_was_truncated,is_valid_review_date,row_quality_flag,silver_timestamp,silver_batch_id,silver_domain
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,0939495848,Treasure Island,null,A3D9OCZKZKH4GK,"Scott A. Reighard ""Scott A. Reighard, Author,...",6/7,4.0,1197504000,Treasure Island,"I have not read this book since I was in 7th grade, but this summer my son wanted to read it. As a 5th grader it was a bit over his head, but we both enjoyed it very much. This book is the reason I became so fascinated with pirates. It's a story that just sticks with you. There are memorable characters and slippery plot complications that make this literally a genuine trip as you turn the pages. Some may consider this an unnecessary read given its time frame, but put in context, this is a delightful story that should resonate with young boys. It's a classic for the very reason that it endures. It finds a place in that little place in the mind called adventure and wanderlust.Jamestown: Journey Back in Time",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating,2007-12-13,2007,12,d0790f3a6cde667685d806cc60d269e3,7,6,0.8571,true,"I have not read this book since I was in 7th grade, but this summer my son wanted to read it. As a 5th grader it was a bit over his head, but we both enjoyed it very much. This book is the reason I became so fascinated with pirates. It's a story that just sticks with you. There are memorable characters and slippery plot complications that make this literally a genuine trip as you turn the pages. Some may consider this an unnecessary read given its time frame, but put in context, this is a delightful story that should resonate with young boys. It's a classic for the very reason that it endures. It finds a place in that little place in the mind called adventure and wanderlust.Jamestown: Journey Back in Time",false,true,CLEAN,2026-04-22T09:54:32.870Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_rating
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,189212419X,Holy Cow! Does God Care about What We Eat?,14.0,A86OQBQ5GH9N1,"Scott Meade ""The Author of Twisted Christians""",5/6,5.0,1276560000,"Holy Cow! Not the Skrimp, too!!!","Hope Egan is a wonderful writer. This book is a must read for everyone who believes that Jesus came to do away with the ""old laws"". There is even a section that details what we should not eat. I must admit that I had a hard time doing away with shrimp and crab legs. I knew it in my heart that I should not have been eating them, but Hope Egan confirmed it for me. And I now buy cheese pizzas and sprinkle them with turkey pepperoni. I would like to thank Hope Egan for writing this book!",s3://retailflow360-capstone3/raw/books/Books_rating.csv,2026-04-22T08:41:21.422Z,2026-04-22,s3://retailflow360-capstone3/raw/books/Books_rating.csv,books_rating,2010-06-15,2010,6,d07e9beeffc953685f96df1858234e14,6,5,0.8333,true,"Hope Egan is a wonderful writer. This book is a must read for everyone who believes that Jesus came to do away with the ""old laws"". There is even a section that details what we should not eat. I must admit that I had a hard time doing away with shrimp and crab legs. I knew it in my heart that I should not have been eating them, but Hope Egan confirmed it for me. And I now buy cheese pizzas and sprinkle them with turkey pepperoni. I would like to thank Hope Egan for writing this book!",false,true,CLEAN,2026-04-22T09:54:32.870Z,086fd8cb-df9c-4da3-b753-ecdf0f61a8fb,books_rating
ccc4ff1d-cec7-4dab-a79f-fbbaf430d6b7,0878422099,Roadside Geology of New Mexico (Roadside Geology Series),null,A28PGP1QWU


Quality distribution:
+--------------------+-------+
|    row_quality_flag|  count|
+--------------------+-------+
|               CLEAN|2431474|
|FLAGGED_ANONYMOUS...| 556865|
|  FLAGGED_NULL_TITLE|    208|
+--------------------+-------+

